# Azure AI Search for RAG: a simple notebook

**What is RAG retrieval?**

1. Split your documents into small pieces called *chunks*.
2. Turn each chunk into numbers (an *embedding*).
3. Store the chunks in Azure AI Search.
4. When a user asks a question, find the best chunks and give them to the LLM.

This notebook builds steps 1 to 4. Run the cells from top to bottom.

**Words you will see**

| Word | Meaning |
|---|---|
| Chunk | A small piece of a document. |
| Embedding / vector | A list of numbers that captures the meaning of text. |
| Index | The Azure table where chunks are stored. |
| HNSW | A graph that finds similar vectors fast. It is approximate. |
| Filter | A rule such as "only tenant acme". |
| Hybrid search | Keyword search + vector search together. |
| Semantic ranker | A second step that re-orders the best results. |

**What you need**

- An Azure AI Search service.
- An Azure OpenAI embedding deployment (1536 dimensions, for example `text-embedding-3-small`).

Create a file named `.env` next to this notebook:

```
AZURE_SEARCH_ENDPOINT=https://<service>.search.windows.net
AZURE_SEARCH_KEY=<admin key>
AOAI_ENDPOINT=https://<resource>.openai.azure.com
AOAI_KEY=<key>
AOAI_EMBED_DEPLOYMENT=<your embedding deployment name>
```

## 1. Setup

In [ ]:
%pip install -q azure-search-documents openai python-dotenv

In [ ]:
import os, re, time, sqlite3, hashlib
from dotenv import load_dotenv

load_dotenv()  # reads the .env file

needed = ["AZURE_SEARCH_ENDPOINT", "AZURE_SEARCH_KEY", "AOAI_ENDPOINT", "AOAI_KEY", "AOAI_EMBED_DEPLOYMENT"]
missing = [name for name in needed if not os.getenv(name)]
assert not missing, f"Please set these environment variables: {missing}"

INDEX_NAME = "rag-chunks"
DIMS = 1536  # must match your embedding model

In [ ]:
from azure.core.credentials import AzureKeyCredential
from azure.search.documents import SearchClient
from azure.search.documents.indexes import SearchIndexClient
from openai import AzureOpenAI

endpoint = os.environ["AZURE_SEARCH_ENDPOINT"]
cred = AzureKeyCredential(os.environ["AZURE_SEARCH_KEY"])

index_client = SearchIndexClient(endpoint, cred)         # creates and deletes indexes
search_client = SearchClient(endpoint, INDEX_NAME, cred)  # reads and writes chunks

aoai = AzureOpenAI(
    azure_endpoint=os.environ["AOAI_ENDPOINT"],
    api_key=os.environ["AOAI_KEY"],
    api_version="2024-10-21",
)

> **Tip for production:** avoid keys. Use `DefaultAzureCredential()` (Microsoft Entra ID) with the roles
> *Search Index Data Contributor* (write) and *Search Index Data Reader* (read).

## 2. Create the index

Each field has a job:

- `searchable`: used for keyword search.
- `filterable`: can be used in a filter. We use this for tenant and group security.
- `key`: the unique id of the chunk.

The vector field stores the embedding. It uses an HNSW graph.

The HNSW numbers below are Azure's defaults. Higher numbers give better recall but are slower.

We also add a *semantic configuration*. The semantic ranker needs it.

In [ ]:
from azure.search.documents.indexes.models import (
    SearchIndex, SearchField, SearchFieldDataType, SimpleField, SearchableField,
    VectorSearch, HnswAlgorithmConfiguration, HnswParameters,
    VectorSearchProfile, VectorSearchAlgorithmMetric,
    SemanticSearch, SemanticConfiguration, SemanticPrioritizedFields, SemanticField,
)

fields = [
    SimpleField(name="chunk_id", type=SearchFieldDataType.String, key=True),
    SimpleField(name="parent_id", type=SearchFieldDataType.String, filterable=True),    # the document id
    SearchableField(name="content", type=SearchFieldDataType.String),                   # the chunk text
    SearchableField(name="section_path", type=SearchFieldDataType.String),              # "Refund Policy > Processing time"
    SimpleField(name="tenant_id", type=SearchFieldDataType.String, filterable=True),    # which customer owns it
    SimpleField(name="source_type", type=SearchFieldDataType.String, filterable=True, facetable=True),
    SimpleField(name="is_current", type=SearchFieldDataType.Boolean, filterable=True),  # False = old version
    SimpleField(name="page_number", type=SearchFieldDataType.Int32, filterable=True),
    SimpleField(name="group_ids", type=SearchFieldDataType.Collection(SearchFieldDataType.String),
                filterable=True),                                                       # who can read it
    SearchField(
        name="content_vector",
        type=SearchFieldDataType.Collection(SearchFieldDataType.Single),
        searchable=True,
        vector_search_dimensions=DIMS,
        vector_search_profile_name="hnsw-profile",
    ),
]

vector_config = VectorSearch(
    algorithms=[HnswAlgorithmConfiguration(
        name="hnsw-cfg",
        parameters=HnswParameters(m=4, ef_construction=400, ef_search=500,
                                  metric=VectorSearchAlgorithmMetric.COSINE),
    )],
    profiles=[VectorSearchProfile(name="hnsw-profile", algorithm_configuration_name="hnsw-cfg")],
)

semantic_config = SemanticSearch(configurations=[
    SemanticConfiguration(
        name="my-semantic-config",
        prioritized_fields=SemanticPrioritizedFields(
            title_field=SemanticField(field_name="section_path"),
            content_fields=[SemanticField(field_name="content")],
        ),
    )
])

index = SearchIndex(name=INDEX_NAME, fields=fields,
                    vector_search=vector_config, semantic_search=semantic_config)
index_client.create_or_update_index(index)
print("Index ready:", INDEX_NAME)

**Good to know**

- You can add new fields later. You cannot change the type or the vector size of an existing field.
- Need exact search? You do not need a second algorithm. Pass `exhaustive=True` when you query (section 8).
- If you delete and recreate the index, also clear the checkpoint table from section 5.

## 3. Chunking

We split markdown at its headings.

Each chunk starts with a *breadcrumb*, like `[Refund Policy > Processing time]`.

The breadcrumb tells the search engine what the chunk is about. A chunk that only says "5 to 7 days" is not useful on its own.

Smaller chunks usually work better than big ones.

In [ ]:
def chunk_markdown(markdown, max_chars=800):
    """Split markdown at headings. Returns a list of {"section_path", "content"}."""
    chunks = []
    path = []   # current heading path, e.g. ["Refund Policy", "Processing time"]
    body = []   # lines of text under the current heading

    def save_chunk():
        text = " ".join(body).strip()
        if text and path:
            section = " > ".join(path)
            for start in range(0, len(text), max_chars):   # cut very long sections
                piece = text[start:start + max_chars]
                chunks.append({"section_path": section, "content": f"[{section}] {piece}"})
        body.clear()

    for line in markdown.splitlines():
        match = re.match(r"^(#{1,6})\s+(.*)", line)
        if match:                                          # this line is a heading
            save_chunk()
            level = len(match.group(1))
            path[:] = path[:level - 1] + [match.group(2).strip()]
        elif line.strip():
            body.append(line.strip())
    save_chunk()
    return chunks


example = "# Refund Policy\n## Processing time\nRefunds take 5 to 7 days.\n## Eligibility\nWithin 30 days."
for chunk in chunk_markdown(example):
    print(chunk)

_Later you can improve this: split by tokens instead of characters, and add a small overlap between chunks._

## 4. Helper functions

Three small helpers:

- `embed`: turns text into vectors. We send texts in small groups to save time.
- `with_retry`: tries again if a call fails (for example, when Azure says "slow down").
- `upload_in_batches`: uploads chunks in groups.

Azure allows up to 1,000 documents or about 16 MB per upload request. Vectors are big, so we use a small batch of 100.

`model=` must be your **deployment name**, not the model name.

In [ ]:
def odata_str(value):
    """Make a string safe for an Azure filter. Doubling the quote blocks filter injection."""
    return "'" + value.replace("'", "''") + "'"


def with_retry(func, tries=3):
    """Run func. If it fails, wait 1s, then 2s, then give up."""
    for attempt in range(tries):
        try:
            return func()
        except Exception as error:
            if attempt == tries - 1:
                raise
            wait = 2 ** attempt
            print(f"  Error: {error}. Retrying in {wait}s...")
            time.sleep(wait)


def embed(texts, batch_size=16):
    vectors = []
    for i in range(0, len(texts), batch_size):
        group = texts[i:i + batch_size]
        response = with_retry(lambda: aoai.embeddings.create(
            model=os.environ["AOAI_EMBED_DEPLOYMENT"], input=group))
        vectors.extend(item.embedding for item in response.data)
    return vectors


def upload_in_batches(rows, batch_size=100):
    for i in range(0, len(rows), batch_size):
        batch = rows[i:i + batch_size]
        results = with_retry(lambda: search_client.upload_documents(documents=batch))
        failed = [r.key for r in results if not r.succeeded]
        if failed:
            raise RuntimeError(f"These chunks failed to upload: {failed}")

## 5. Safe ingestion: what if it stops halfway?

Big jobs fail sometimes: network errors, rate limits, a closed laptop.

Azure does **not** roll back. Chunks already uploaded stay in the index. A document can end up half uploaded.

So we need two things.

**1. Safe to repeat.** Every chunk has a fixed id (`doc_id-0`, `doc_id-1`, ...). Uploading it again just replaces it. No duplicates.

**2. A checkpoint table.** It lives outside the index. It has one row per document:

| doc_id | doc_hash | status | chunk_count |
|---|---|---|---|
| acme-refund | 9f2c... | done | 2 |
| acme-salary | 41ab... | failed | |

The rules:

- A document is `done` only after **all** its chunks are uploaded.
- On restart, skip documents that are `done` and unchanged.
- Redo everything else.

The hash covers the text **and** the permissions. If someone changes who can read a document, we must re-ingest it.

This is how most teams do it. Bigger systems use a queue with retries. If your data is in Blob, SQL or Cosmos DB, an Azure *indexer* keeps its own checkpoint for you.

In [ ]:
db = sqlite3.connect("ingest_checkpoint.db")   # a small file, easy to inspect
db.execute("""CREATE TABLE IF NOT EXISTS checkpoint (
    doc_id TEXT PRIMARY KEY, doc_hash TEXT, status TEXT, chunk_count INTEGER, error TEXT)""")


def get_row(doc_id):
    return db.execute("SELECT doc_hash, status FROM checkpoint WHERE doc_id=?", (doc_id,)).fetchone()


def save_row(doc_id, doc_hash, status, chunk_count=None, error=None):
    db.execute("INSERT OR REPLACE INTO checkpoint VALUES (?, ?, ?, ?, ?)",
               (doc_id, doc_hash, status, chunk_count, error))
    db.commit()


def show_checkpoint():
    for row in db.execute("SELECT doc_id, status, chunk_count, error FROM checkpoint"):
        print(row)


def reset_checkpoint():
    db.execute("DELETE FROM checkpoint")
    db.commit()

Now the main function, `ingest`. Read it in order:

1. Skip if already done.
2. Mark as `in_progress`.
3. Chunk, embed, upload.
4. Delete old chunks that no longer exist (for example, the new version is shorter).
5. Mark as `done`.

We **upload first and delete after**. If we crashed in between, the document would still be searchable.

In [ ]:
CRASH_ON = None   # for the demo below: the doc id where we pretend to crash


def delete_stale_chunks(doc_id, keep_ids):
    found = search_client.search(search_text="*", filter=f"parent_id eq {odata_str(doc_id)}",
                                 select=["chunk_id"], top=1000)
    stale = [{"chunk_id": r["chunk_id"]} for r in found if r["chunk_id"] not in keep_ids]
    if stale:
        search_client.delete_documents(documents=stale)


def ingest(doc):
    doc_id = doc["doc_id"]
    fingerprint = doc["markdown"] + doc["tenant_id"] + str(doc["group_ids"]) + str(doc["is_current"])
    doc_hash = hashlib.sha256(fingerprint.encode()).hexdigest()

    row = get_row(doc_id)
    if row and row[0] == doc_hash and row[1] == "done":
        print(f"{doc_id}: skipped (already done)")
        return

    save_row(doc_id, doc_hash, "in_progress")
    try:
        chunks = chunk_markdown(doc["markdown"])
        vectors = embed([c["content"] for c in chunks])
        rows = [{
            "chunk_id": f"{doc_id}-{i}",
            "parent_id": doc_id,
            "content": c["content"],
            "section_path": c["section_path"],
            "tenant_id": doc["tenant_id"],
            "source_type": doc["source_type"],
            "is_current": doc["is_current"],
            "page_number": 1,
            "group_ids": doc["group_ids"],
            "content_vector": vector,
        } for i, (c, vector) in enumerate(zip(chunks, vectors))]

        upload_in_batches(rows)
        delete_stale_chunks(doc_id, keep_ids=[r["chunk_id"] for r in rows])
        if doc_id == CRASH_ON:
            raise RuntimeError("Simulated crash")
        save_row(doc_id, doc_hash, "done", chunk_count=len(rows))
        print(f"{doc_id}: done ({len(rows)} chunks)")
    except Exception as error:
        save_row(doc_id, doc_hash, "failed", error=str(error))
        raise


def run_ingestion(docs):
    for doc in docs:
        ingest(doc)

### Sample data

Five small documents. They help us test filters later.

| doc | tenant | groups | current? | why |
|---|---|---|---|---|
| `acme-refund` | acme | eng, hr | yes | the right answer |
| `acme-refund-old` | acme | eng, hr | **no** | old version, must be hidden |
| `acme-salary` | acme | **hr** | yes | an `eng` user must not see it |
| `globex-refund` | **globex** | eng | yes | another customer, must not leak |
| `tiny-faq` | tiny-tenant | eng | yes | a very small set, for the exact scan |

In [ ]:
REFUND_NEW = """# Refund Policy
## Processing time
Refunds are processed within 5 to 7 business days after approval. The money returns to the original payment method.
## Eligibility
Items can be returned within 30 days of delivery if unused and in original packaging.
"""
REFUND_OLD = "# Refund Policy\n## Processing time\nRefunds are processed within 14 business days after approval.\n"
SALARY = "# Compensation\n## Salary bands\nSalary bands are confidential and visible to the HR group only.\n"
GLOBEX = "# Refund Policy\n## Processing time\nGlobex refunds take up to 30 days to complete.\n"
TINY = "# FAQ\n## Refunds\nRefunds for Tiny Tenant customers take 2 days.\n"

DOCS = [
    dict(doc_id="acme-refund",     markdown=REFUND_NEW, tenant_id="acme",        group_ids=["eng", "hr"], source_type="policy", is_current=True),
    dict(doc_id="acme-refund-old", markdown=REFUND_OLD, tenant_id="acme",        group_ids=["eng", "hr"], source_type="policy", is_current=False),
    dict(doc_id="acme-salary",     markdown=SALARY,     tenant_id="acme",        group_ids=["hr"],        source_type="hr",     is_current=True),
    dict(doc_id="globex-refund",   markdown=GLOBEX,     tenant_id="globex",      group_ids=["eng"],       source_type="policy", is_current=True),
    dict(doc_id="tiny-faq",        markdown=TINY,       tenant_id="tiny-tenant", group_ids=["eng"],       source_type="faq",    is_current=True),
]

### Demo: crash, then restart

First run: we pretend to crash while ingesting `acme-salary`.

Look at the checkpoint after the crash. Two documents are `done`, one is `failed`, two have not started.

In [ ]:
CRASH_ON = "acme-salary"
try:
    run_ingestion(DOCS)
except Exception as error:
    print("\nIngestion stopped:", error)

print("\nCheckpoint table:")
show_checkpoint()

Now restart. We run the **same code** again.

Done documents are skipped. The failed one is redone. The rest are processed.

In [ ]:
CRASH_ON = None
run_ingestion(DOCS)

print("\nCheckpoint table:")
show_checkpoint()

### Check the index matches the checkpoint

Azure indexes in near real time, so we wait a few seconds first.

For each `done` document, we count its chunks in the index. The count should match the checkpoint. A mismatch means something is missing, and you can re-ingest that document.

In [ ]:
time.sleep(3)

def count_in_index(doc_id):
    result = search_client.search(search_text="*", filter=f"parent_id eq {odata_str(doc_id)}",
                                  include_total_count=True, top=0)
    return result.get_count()


def verify_all():
    for doc_id, expected in db.execute("SELECT doc_id, chunk_count FROM checkpoint WHERE status='done'"):
        actual = count_in_index(doc_id)
        print(f"{doc_id:<18} expected={expected} in_index={actual}  {'OK' if actual == expected else 'MISMATCH'}")


verify_all()

## 6. The security filter

Every query carries a filter. Build it on the server, from the logged-in user. Never build it from text the user typed.

Our filter has three rules:

1. Same tenant.
2. Current version only.
3. The user is in at least one allowed group.

If the user has no groups, the filter matches nothing. That is the safe default.

In [ ]:
from azure.search.documents.models import VectorizedQuery, VectorFilterMode


def build_filter(tenant_id, user_groups):
    groups = ",".join(user_groups)
    return (f"tenant_id eq {odata_str(tenant_id)} and is_current eq true "
            f"and group_ids/any(g: search.in(g, {odata_str(groups)}, ','))")


def show(results):
    for r in results:
        rerank = r.get("@search.reranker_score")
        rerank = f"{rerank:.2f}" if rerank is not None else "-"
        print(f"score={r['@search.score']:.3f}  rerank={rerank:<5} {r['chunk_id']:<16} {r['content'][:60]}")


QUESTION = "how long do refunds take?"
FILTER = build_filter("acme", ["eng"])
print(FILTER)

## 7. Three ways to search

### 7a. Vector search

Turn the question into a vector. Find the closest chunks.

Good at meaning. Weaker at exact words like product codes.

- `k` = how many neighbours to find.
- `top` = how many results to return.

In [ ]:
def vector_only_search(question, flt, k=10, mode=VectorFilterMode.PRE_FILTER, exhaustive=False):
    vq = VectorizedQuery(vector=embed([question])[0], k_nearest_neighbors=k,
                         fields="content_vector", exhaustive=exhaustive)
    return list(search_client.search(
        search_text=None,                 # no keyword search
        vector_queries=[vq],
        filter=flt,
        vector_filter_mode=mode,
        select=["chunk_id", "content", "section_path"],
        top=k,
    ))


results = vector_only_search(QUESTION, FILTER)
show(results)

# Safety check: nothing from another tenant, an old version, or an HR-only doc
bad_prefixes = ("globex", "acme-salary", "acme-refund-old")
assert not any(r["chunk_id"].startswith(bad_prefixes) for r in results)
print("Filter check passed")

### 7b. Hybrid search

Run keyword search and vector search together. Azure merges the two lists with *Reciprocal Rank Fusion (RRF)*.

RRF uses the **rank** of each result, not the score. This works because keyword scores and vector scores are on different scales.

The filter applies to both parts.

In [ ]:
def hybrid_search(question, flt, k=50, top=10):
    vq = VectorizedQuery(vector=embed([question])[0], k_nearest_neighbors=k, fields="content_vector")
    return list(search_client.search(
        search_text=question,             # keyword part
        vector_queries=[vq],              # vector part
        filter=flt,
        vector_filter_mode=VectorFilterMode.PRE_FILTER,
        select=["chunk_id", "content", "section_path"],
        top=top,
    ))


show(hybrid_search(QUESTION, FILTER))

### 7c. Hybrid + semantic ranker

The semantic ranker re-reads the best results (up to 50) and puts the most relevant first. It adds a `rerank` score from 0 to 4.

Microsoft's own 2023 test found hybrid + semantic ranker gave the best results. Keyword-only and vector-only were clearly worse.

The semantic ranker must be enabled on your search service. It has a free monthly quota.

Remember the flow: **find many (hybrid), then re-rank few (semantic), then send the top few to the LLM.**

In [ ]:
def hybrid_semantic_search(question, flt, k=50, top=10):
    vq = VectorizedQuery(vector=embed([question])[0], k_nearest_neighbors=k, fields="content_vector")
    return list(search_client.search(
        search_text=question,
        vector_queries=[vq],
        filter=flt,
        vector_filter_mode=VectorFilterMode.PRE_FILTER,
        query_type="semantic",                      # turns the semantic ranker on
        semantic_configuration_name="my-semantic-config",
        select=["chunk_id", "content", "section_path"],
        top=top,
    ))


show(hybrid_semantic_search(QUESTION, FILTER))

## 8. Filters and the HNSW graph

**Does a filter damage the graph?** No.

The graph is built when you upload vectors. A filter is only a rule used at query time. It never changes the graph.

**How is the filter applied?** Azure splits the index into *shards*. It searches the graph on each shard.

- **`preFilter`** checks the filter *while* walking the graph. It keeps walking until it finds `k` matches. Best recall. It can be slower when the filter matches very few chunks.
- **`postFilter`** walks the graph and filters the results, on each shard. It can miss matches and return fewer than `k`.

Use `preFilter`. It is the recommended default. (It is the default for indexes created after October 2023.)

**Very selective filter?** For example, a tiny tenant. Use `exhaustive=True`. Azure then compares the query with every matching vector. Recall is 100%. This is fast when only a few vectors match.

### Measure recall

*Recall* = of the true best `k` chunks, how many did HNSW find?

We get the "true" list with an exact search (`exhaustive=True`) and the same filter.

> With five tiny documents, both modes show 1.00. The difference appears on large indexes with many tenants. The code is what matters here.

In [ ]:
def filtered_recall(question, flt, k=10, mode=VectorFilterMode.PRE_FILTER):
    approx = {r["chunk_id"] for r in vector_only_search(question, flt, k=k, mode=mode)}
    exact = {r["chunk_id"] for r in vector_only_search(question, flt, k=k, exhaustive=True)}
    return len(approx), len(exact), len(approx & exact) / max(1, len(exact))


for mode in (VectorFilterMode.PRE_FILTER, VectorFilterMode.POST_FILTER):
    returned, exact_count, recall = filtered_recall(QUESTION, FILTER, mode=mode)
    print(f"{mode.value:<10} returned={returned}  exact={exact_count}  recall={recall:.2f}")

### Exact scan on a tiny tenant

In [ ]:
tiny_filter = build_filter("tiny-tenant", ["eng"])
show(vector_only_search(QUESTION, tiny_filter, k=10, exhaustive=True))

## 9. Measure your search quality

Do not guess which search is better. Measure it.

1. Write a list of questions.
2. For each, write the chunk that should come first.
3. Count how often each method puts it first. This is called *hit@1*.

Our test set is tiny, so scores will be high. In real projects, use 50 or more real questions. Re-run this test after every change (chunk size, search type, reranker).

In [ ]:
TEST_SET = [
    ("how long do refunds take?",              "acme-refund-0"),
    ("which payment method gets the money?",   "acme-refund-0"),
    ("can I return an item after 3 weeks?",    "acme-refund-1"),
    ("what condition must returned items be?", "acme-refund-1"),
]


def hit_rate(search_fn):
    hits = 0
    for question, expected_id in TEST_SET:
        results = search_fn(question, FILTER)
        first = results[0]["chunk_id"] if results else None
        hits += (first == expected_id)
    return hits / len(TEST_SET)


methods = {
    "vector":          lambda q, f: vector_only_search(q, f, k=3),
    "hybrid":          hybrid_search,
    "hybrid+semantic": hybrid_semantic_search,
}

for name, search_fn in methods.items():
    try:
        print(f"{name:<16} hit@1 = {hit_rate(search_fn):.2f}")
    except Exception as error:
        print(f"{name:<16} skipped: {error}")

## 10. What teams do in practice

### Retrieval

- **Use hybrid search + semantic ranker** as the starting point.
- **Keep chunks small** and add the heading path to each chunk.
- **Filter on the server** for tenant and permissions.
- **Measure** with a labelled question set, and test one change at a time.
- **Consider query rewriting** (for example, turn "and the second one?" into a full question) before searching.
- **Newer option:** Microsoft now recommends *agentic retrieval* for new RAG projects. An LLM plans the searches for you. This notebook shows the classic pattern, which is the right thing to learn first.

### Ingestion

- **Fixed chunk ids and upserts**, so re-running is safe.
- **A checkpoint table**, so a restart skips finished work.
- **Hash the content and permissions**, so only changed documents are redone.
- **Batch** embeddings and uploads. **Retry** with a short wait.
- **Upload first, delete old chunks after.**
- **Verify** the index against the checkpoint.
- **At scale:** a queue with workers and a dead-letter queue, or an Azure *indexer* for Blob, SQL or Cosmos DB data.
- **Rebuilding everything?** Build a new index, test it, then switch to it.

## Cleanup (optional)

In [ ]:
# reset_checkpoint()                 # forget all progress (use this if you delete the index)
# index_client.delete_index(INDEX_NAME)   # delete the index to stop paying for it

## Interview cheat sheet

**Why hybrid and not only vector?**
Vectors find meaning but blur exact words (codes, names). Keywords find exact words but miss paraphrases. Hybrid gets both.

**What is RRF?**
It merges ranked lists using each result's rank, not its score. Score = sum of 1 / (k + rank).

**What does the semantic ranker do?**
It re-orders the top results (up to 50) with a deep-learning model. It needs a semantic configuration on the index.

**`k` vs `top`?**
`k` is how many vector neighbours to find. `top` is how many results to return. Use a big `k` and a small `top`.

**Do filters break HNSW?**
No. The graph is built at upload time. Filters only apply at query time.

**preFilter vs postFilter?**
`preFilter` checks the filter while walking the graph and keeps going until it has `k` matches. `postFilter` filters afterwards and may return fewer than `k`.

**HNSW vs exact search?**
HNSW is fast but approximate. Exact (`exhaustive=True`) is slow but perfect. Use exact for tiny filtered sets and for ground truth.

**How do you secure data by tenant or group?**
Filterable `tenant_id` and `group_ids` fields. Build the filter on the server from the user's identity. Escape quotes.

**Ingestion stops halfway. What now?**
Azure does not roll back. Re-run the same job. Fixed chunk ids make it safe. A checkpoint table skips finished documents.

**How do you know what is already in the index?**
Keep a checkpoint table outside the index. Verify by counting chunks per document. Do not rely on the index alone.

**Is one-by-one upload bad?**
It is slow: one network call per document, and more throttling. Batch up to 1,000 documents or about 16 MB per request. Test batch sizes for your data.

**What is the slowest part of a huge ingestion?**
Usually embeddings. They are limited by your Azure OpenAI quota (tokens per minute).

**How do you evaluate retrieval?**
A labelled question set. Measure hit@1, recall@k or NDCG before and after each change.

**Gotchas**
Vector size is fixed once the index is created. A new embedding model means re-embedding everything. Indexing takes a moment to show up. Keep the checkpoint and the index in sync.

---

**Sources used for this version:** Microsoft Learn pages "Add a filter to a vector query in Azure AI Search", "Index large data sets in Azure AI Search", "Run or reset indexers, skills, or documents", "Retrieval augmented generation overview in Azure AI Search", the Azure Architecture Center guide "Designing and developing a RAG solution", and the Microsoft blog "Azure AI Search: Outperforming vector search with hybrid retrieval and reranking" (2023).